# Summary_Day9.ipynb  
## 다중 분류 · Softmax · CrossEntropyLoss · torch.max · NLLLoss · KL Divergence · 다층 신경망

이번 9강은 **다중 분류 Multi-class Classification**를 정리하는 강의다.

8강에서 이진 분류를 배웠다면, 9강은 선택지가 2개에서 여러 개로 늘어나는 구조다.

```text
이진 분류: 0 또는 1 중 하나를 고른다
다중 분류: 0, 1, 2, ..., N-1 중 하나를 고른다
```

이번 강의의 전체 흐름은 다음이다.

```text
다중 분류 문제 정의
→ 출력 차원이 1개에서 N개로 늘어남
→ 가중치 벡터가 가중치 행렬로 바뀜
→ Softmax로 N개 점수를 확률 분포로 변환
→ CrossEntropyLoss가 Softmax + Log + NLLLoss를 내부에서 처리
→ torch.max(outputs, 1)[1]로 예측 class 선택
→ Iris 데이터로 선형 다중 분류 실습
→ 입력 feature를 2개에서 4개로 늘려 비교
→ Wine 데이터로 다층 신경망 다중 분류 실습
→ NLLLoss와 KL Divergence 개념 정리
```

> 필기 포인트:  
> 다중 분류에서는 모델 출력이 확률 1개가 아니라, class 개수만큼의 점수 vector다.  
> PyTorch의 표준 패턴은 **모델은 logits를 그대로 출력하고, 손실함수는 `nn.CrossEntropyLoss()`를 쓰는 것**이다.

## 1. 라이브러리 준비

이번 실습에서는 NumPy, Matplotlib, PyTorch, scikit-learn을 사용한다.

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
```

- `torch`: Tensor와 자동 미분을 사용할 때 사용한다.
- `nn`: `Linear`, `ReLU`, `CrossEntropyLoss`, `NLLLoss` 등을 사용할 때 필요하다.
- `optim`: SGD, AdamW 같은 Optimizer를 사용할 때 필요하다.
- `load_iris`: Iris 다중 분류 데이터를 불러온다.
- `load_wine`: Wine 다중 분류 데이터를 불러온다.
- `StandardScaler`: feature scale을 맞출 때 사용한다.
- `classification_report`: class별 precision, recall, f1-score를 확인한다.

> 실습 메모:  
> 원본 노트북에는 Colab 폰트 설치와 torchviz/torchinfo 코드가 있다.  
> 이 Summary는 바로 실행되도록 외부 설치가 필요한 코드는 제외하고 핵심 다중 분류 흐름만 정리한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("PyTorch:", torch.__version__)

## 2. 이진 분류와 다중 분류 차이

이진 분류는 선택지가 2개인 문제다.

```text
정상 / 사기
스팸 / 정상
질병 / 정상
```

다중 분류는 선택지가 3개 이상인 문제다.

```text
setosa / versicolor / virginica
숫자 0~9
뉴스 카테고리 여러 개
```

가장 큰 차이는 모델 출력 차원이다.

| 문제 | 출력 형태 |
|---|---|
| 이진 분류 | 출력 1개 |
| 다중 분류 | class 개수만큼 출력 |

In [ ]:
classification_summary = {
    "Binary Classification": "출력 1개로 0/1을 판단한다",
    "Multi-class Classification": "class 개수만큼 출력하고 가장 큰 class를 고른다"
}

for key, value in classification_summary.items():
    print(f"{key}: {value}")

> 기억할 점:  
> 다중 분류 모델의 출력 shape은 보통 `[batch_size, num_classes]`다.  
> 예를 들어 75개 데이터를 3개 class로 분류하면 출력 shape은 `[75, 3]`이다.

## 3. 출력 차원: 1개에서 N개로

이진 분류에서는 class 1일 확률 하나만 보면 됐다.

다중 분류에서는 각 class별 점수가 모두 필요하다.

예를 들어 Iris 3개 품종을 분류한다면 출력은 3개다.

```text
[setosa 점수, versicolor 점수, virginica 점수]
```

이 점수는 아직 확률이 아니다.  
이 raw score를 **logits**라고 부른다.

In [ ]:
logits_example = torch.tensor([
    [2.1, 0.8, -1.3],
    [0.2, 2.7, 0.4],
    [-0.5, 0.3, 2.2]
])

print("logits:")
print(logits_example)
print("shape:", logits_example.shape)

출력 해석:

```text
shape = [3, 3]
```

- 첫 번째 3은 샘플 개수다.
- 두 번째 3은 class 개수다.
- 각 행은 한 샘플의 class별 점수다.

## 4. 가중치 벡터에서 가중치 행렬로

이진 분류에서는 출력이 1개라서 하나의 가중치 벡터로 충분했다.

다중 분류에서는 출력이 class 개수만큼 필요하다.  
그래서 여러 개의 가중치 벡터가 모인 **가중치 행렬**이 필요하다.

PyTorch의 `nn.Linear(2, 3)`은 다음 의미다.

```text
입력 feature 2개
출력 class 점수 3개
```

### 함수 사용법

```python
nn.Linear(in_features, out_features)
```

- `in_features`: 입력 feature 개수다.
- `out_features`: 출력 class 개수다.

In [ ]:
linear_multi = nn.Linear(2, 3)

print(linear_multi)
print("weight shape:", linear_multi.weight.shape)
print("bias shape:", linear_multi.bias.shape)

출력 해석:

```text
weight shape = [3, 2]
bias shape = [3]
```

- class 3개를 위한 weight row가 3개 있다.
- 각 row는 해당 class 점수를 계산하는 하나의 분류기처럼 볼 수 있다.

> 필기 포인트:  
> 다중 분류는 “여러 개의 작은 분류기가 동시에 점수를 내고, 가장 큰 점수를 고르는 구조”로 이해하면 편하다.

## 5. Softmax 함수

Softmax는 여러 개의 logits를 확률 분포로 바꾸는 함수다.

특징은 다음이다.

```text
각 확률은 0과 1 사이
모든 확률의 합은 1
큰 logit일수록 큰 확률
```

### 함수 사용법

```python
torch.softmax(logits, dim=1)
```

- `logits`: 모델의 raw score다.
- `dim=1`: class 방향으로 softmax를 적용한다.

In [ ]:
probs_example = torch.softmax(logits_example, dim=1)

print("probabilities:")
print(probs_example)

print("각 행의 합:")
print(probs_example.sum(dim=1))

> 시험 포인트:  
> 다중 분류에서는 보통 `dim=1`을 사용한다.  
> 출력 shape이 `[batch, class]`이기 때문에 class 방향이 두 번째 차원이다.

In [ ]:
sample_logits = torch.tensor([[2.1, 0.8, -1.3]])
sample_probs = torch.softmax(sample_logits, dim=1)

plt.bar(["class 0", "class 1", "class 2"], sample_probs.numpy().ravel())
plt.ylabel("probability")
plt.title("Softmax Probability Distribution")
plt.show()

그래프 해석:

- 가장 큰 logit을 가진 class가 가장 큰 확률을 가진다.
- 그래도 나머지 class의 확률이 완전히 0이 되지는 않는다.
- 그래서 softmax는 hard max가 아니라 soft한 확률 분포다.

## 6. CrossEntropyLoss의 PyTorch 표준 패턴

이론적으로 다중 분류 손실은 다음 흐름이다.

```text
logits
→ Softmax
→ log
→ 정답 class 위치만 추출
→ 음수로 바꿔 loss 계산
```

하지만 PyTorch에서는 이 과정을 한 번에 처리한다.

### 함수 사용법

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
```

- `logits`: softmax 전 raw score다.
- `labels`: one-hot이 아니라 class index다.
- `labels` dtype은 반드시 `torch.long`이다.

> 핵심:  
> `CrossEntropyLoss`에 넣기 전에 Softmax를 직접 적용하지 않는다.

In [ ]:
labels_example = torch.tensor([0, 1, 2])

criterion_ce = nn.CrossEntropyLoss()

loss_ce = criterion_ce(logits_example, labels_example)

print("CrossEntropyLoss:", loss_ce.item())

## 7. CrossEntropyLoss 직접 분해하기

`CrossEntropyLoss`는 내부적으로 다음과 같은 일을 한다.

```text
log_softmax + NLLLoss
```

직접 계산해서 같은 결과인지 확인한다.

In [ ]:
log_probs = F.log_softmax(logits_example, dim=1)

nll_loss = nn.NLLLoss()

loss_nll = nll_loss(log_probs, labels_example)

print("CrossEntropyLoss:", loss_ce.item())
print("LogSoftmax + NLLLoss:", loss_nll.item())

출력 해석:

- 두 값이 같거나 거의 같다.
- 그래서 PyTorch의 표준 방식은 `logits + CrossEntropyLoss`다.
- `LogSoftmax + NLLLoss`는 같은 원리를 직접 분리해서 쓴 방식이다.

## 8. NLLLoss란 무엇인가

NLLLoss는 Negative Log Likelihood Loss다.

다중 분류에서 하는 일은 단순하다.

```text
각 샘플에서 정답 class에 해당하는 log probability만 뽑는다
그 값에 음수를 붙인다
평균을 낸다
```

예를 들어 정답 label이 `[0, 1, 2]`이면 다음 값을 뽑는다.

```text
sample 0 → class 0의 log probability
sample 1 → class 1의 log probability
sample 2 → class 2의 log probability
```

In [ ]:
print("log_probs:")
print(log_probs)

picked_log_probs = log_probs[torch.arange(len(labels_example)), labels_example]

manual_nll = -picked_log_probs.mean()

print("정답 class log probability:")
print(picked_log_probs)

print("직접 계산 NLL:", manual_nll.item())
print("nn.NLLLoss:", loss_nll.item())

> 필기 포인트:  
> NLLLoss는 “정답 위치만 뽑아서 벌점을 계산하는 함수”라고 생각하면 된다.  
> PyTorch는 one-hot label 없이 정수 label만으로 이 위치를 바로 찾는다.

## 9. softmax(dim=0)과 softmax(dim=1) 차이

다중 분류에서 softmax 차원을 잘못 잡으면 해석이 완전히 달라진다.

출력 shape이 `[batch, class]`일 때는 class 방향인 `dim=1`을 써야 한다.

In [ ]:
o3 = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],
    [1.5, 3.0, 0.2, 0.8],
    [0.3, 0.7, 2.5, 1.0]
])

softmax_dim0 = torch.softmax(o3, dim=0)
softmax_dim1 = torch.softmax(o3, dim=1)

print("원본 shape:", o3.shape)

print("\nsoftmax dim=0:")
print(softmax_dim0)
print("열 합:", softmax_dim0.sum(dim=0))

print("\nsoftmax dim=1:")
print(softmax_dim1)
print("행 합:", softmax_dim1.sum(dim=1))

해석:

- `dim=0`은 열 방향으로 합이 1이 된다.
- `dim=1`은 각 샘플의 class 확률 합이 1이 된다.
- 다중 분류 예측 확률로 해석하려면 보통 `dim=1`이 맞다.

## 10. torch.max로 예측 class 구하기

다중 분류 모델 출력은 `[batch, class]` 형태의 logits다.

예측 class는 각 행에서 가장 큰 값의 index다.

### 함수 사용법

```python
values, indices = torch.max(outputs, dim=1)
pred = torch.max(outputs, 1)[1]
```

- `values`: 각 행의 최댓값이다.
- `indices`: 각 행에서 최댓값이 있는 위치다.
- `indices`가 예측 class label이다.

In [ ]:
values, indices = torch.max(logits_example, dim=1)

print("values:", values)
print("indices:", indices)

pred_example = torch.max(logits_example, 1)[1]
print("pred:", pred_example)

> 기억할 점:  
> 다중 분류 예측에서는 보통 `torch.max(outputs, 1)[1]` 또는 `outputs.argmax(dim=1)`을 쓴다.

## 11. Iris 다중 분류 데이터 준비

이번에는 Iris 3개 품종을 모두 사용한다.

원본 데이터는 다음 구조다.

```text
데이터 150개
feature 4개
class 3개
```

기본 실습에서는 시각화를 쉽게 하기 위해 feature 2개만 사용한다.

```text
sepal length
petal length
```

In [ ]:
iris = load_iris()

x_org = iris.data
y_org = iris.target

print("원본 데이터:", x_org.shape, y_org.shape)
print("class 이름:", iris.target_names)
print("feature 이름:", iris.feature_names)
print("class 분포:", np.bincount(y_org))

## 12. feature 2개 선택하기

시각화를 위해 4개 feature 중 2개만 고른다.

원본 강의 흐름처럼 다음 두 feature를 사용한다.

```text
0번: sepal length
2번: petal length
```

In [ ]:
x_select = x_org[:, [0, 2]]

print("x_select shape:", x_select.shape)
print(x_select[:5])

## 13. Train / Validation 분할

150개 데이터를 train과 validation으로 나눈다.

### 함수 사용법

```python
train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)
```

- `test_size=0.5`: 절반을 validation으로 사용한다.
- `stratify=y`: class 비율을 유지한다.
- `random_state`: 결과 재현을 위해 랜덤을 고정한다.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_select,
    y_org,
    test_size=0.5,
    random_state=42,
    stratify=y_org
)

print("x_train:", x_train.shape)
print("x_val:", x_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

print("train class 분포:", np.bincount(y_train))
print("val class 분포:", np.bincount(y_val))

## 14. Iris 산점도 확인

모델을 만들기 전에 class별 데이터 분포를 먼저 본다.

In [ ]:
for class_id, class_name in enumerate(iris.target_names):
    class_data = x_train[y_train == class_id]
    plt.scatter(class_data[:, 0], class_data[:, 1], label=f"{class_id}: {class_name}")

plt.xlabel("sepal length")
plt.ylabel("petal length")
plt.title("Iris Train Data: 3 Classes")
plt.legend()
plt.show()

그래프 해석:

- class 0은 비교적 잘 분리되어 있다.
- class 1과 class 2는 일부 겹치는 영역이 있다.
- 선형 모델은 직선 경계로 분류하려고 하기 때문에 완벽히 나누기 어려운 부분이 생길 수 있다.

## 15. Tensor 변환

다중 분류에서 입력과 정답 Tensor 타입이 중요하다.

```python
inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).long()
```

- 입력은 실수형 `float`이다.
- 정답 label은 정수형 `long`이다.
- `CrossEntropyLoss`는 정답 label을 `long` 타입 class index로 받는다.

In [ ]:
inputs = torch.tensor(x_train).float().to(device)
labels = torch.tensor(y_train).long().to(device)

inputs_val = torch.tensor(x_val).float().to(device)
labels_val = torch.tensor(y_val).long().to(device)

print("inputs:", inputs.shape, inputs.dtype)
print("labels:", labels.shape, labels.dtype)
print("inputs_val:", inputs_val.shape, inputs_val.dtype)
print("labels_val:", labels_val.shape, labels_val.dtype)

> 시험 포인트:  
> 이진 분류의 BCE 계열은 label을 float로 쓰는 경우가 많다.  
> 다중 분류의 CrossEntropyLoss는 label을 long으로 써야 한다.

## 16. 선형 다중 분류 모델 정의

Iris feature 2개를 입력받아 class 3개 점수를 출력하는 모델을 만든다.

```text
입력 2개 → 출력 3개
```

모델 마지막에 Softmax를 붙이지 않는다.  
`CrossEntropyLoss`가 내부에서 처리하기 때문이다.

In [ ]:
class MultiClassLinearNet(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        return self.l1(x)

n_input = inputs.shape[1]
n_output = len(np.unique(y_org))

net = MultiClassLinearNet(n_input, n_output).to(device)

print(net)
print("weight shape:", net.l1.weight.shape)
print("bias shape:", net.l1.bias.shape)

## 17. 손실함수와 Optimizer 정의

다중 분류의 표준 손실함수는 `CrossEntropyLoss`다.

### 함수 사용법

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)
```

- `criterion`: logits와 class index label을 비교한다.
- `optimizer`: 파라미터를 업데이트한다.
- `lr`: learning rate다.

In [ ]:
criterion = nn.CrossEntropyLoss()

lr = 0.01
optimizer = optim.SGD(net.parameters(), lr=lr)

print("criterion:", criterion)
print("optimizer:", optimizer)

## 18. 학습 전 출력과 예측 확인

학습 전에 모델 출력 shape과 예측 class를 확인한다.

In [ ]:
outputs = net(inputs)
loss = criterion(outputs, labels)

print("outputs shape:", outputs.shape)
print("loss:", loss.item())

print("\ntorch.max(outputs, 1):")
print(torch.max(outputs, 1))

pred = torch.max(outputs, 1)[1]
acc = (pred == labels).float().mean().item()

print("\n초기 accuracy:", acc)

출력 해석:

- `outputs shape = [75, 3]`이면 75개 샘플, 3개 class 점수다.
- `torch.max(outputs, 1)[1]`이 예측 class다.
- 초기 정확도는 아직 학습 전이므로 낮을 수 있다.

## 19. 학습 함수 만들기

학습 루프를 함수로 만든다.

다중 분류 학습의 기본 순서는 다음이다.

```text
zero_grad
→ outputs = model(inputs)
→ loss = CrossEntropyLoss(outputs, labels)
→ loss.backward()
→ optimizer.step()
→ accuracy 기록
```

In [ ]:
def evaluate_multiclass(model, inputs, labels, criterion):
    model.eval()

    with torch.no_grad():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        pred = torch.max(outputs, 1)[1]
        acc = (pred == labels).float().mean()

    return loss.item(), acc.item()


def train_linear_multiclass(model, inputs, labels, inputs_val, labels_val,
                            criterion, optimizer, num_epochs=1000, print_interval=50):
    history = []

    for epoch in range(num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        if epoch % print_interval == 0:
            train_loss, train_acc = evaluate_multiclass(model, inputs, labels, criterion)
            val_loss, val_acc = evaluate_multiclass(model, inputs_val, labels_val, criterion)

            history.append([epoch, train_loss, train_acc, val_loss, val_acc])

    return np.array(history)

history = train_linear_multiclass(
    net,
    inputs,
    labels,
    inputs_val,
    labels_val,
    criterion,
    optimizer,
    num_epochs=1000,
    print_interval=50
)

print("초기 val acc:", history[0, 4])
print("최종 val acc:", history[-1, 4])

### 함수 사용법 정리

```python
model.train()
```

- 모델을 학습 모드로 둔다.

```python
model.eval()
```

- 모델을 평가 모드로 둔다.

```python
with torch.no_grad():
```

- 평가할 때 gradient 계산을 끈다.

## 20. 학습 곡선 확인

Loss와 Accuracy가 epoch에 따라 어떻게 변하는지 확인한다.

In [ ]:
plt.plot(history[:, 0], history[:, 1], label="train loss")
plt.plot(history[:, 0], history[:, 3], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Iris Multi-class Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history[:, 0], history[:, 2], label="train acc")
plt.plot(history[:, 0], history[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Iris Multi-class Accuracy")
plt.legend()
plt.show()

그래프 해석:

- 이상적인 손실 곡선은 점점 감소한다.
- 이상적인 정확도 곡선은 점점 증가한다.
- train과 validation 곡선이 비슷하게 움직이면 학습이 비교적 안정적이다.

## 21. 모델 출력값을 Softmax로 해석하기

`CrossEntropyLoss`를 사용했기 때문에 모델 출력은 확률이 아니라 logits다.

확률로 보고 싶을 때는 직접 Softmax를 적용한다.

In [ ]:
net.eval()

indices_to_check = [0, 1, 2, 3, 4]

with torch.no_grad():
    sample_inputs = inputs_val[indices_to_check]
    sample_logits = net(sample_inputs)
    sample_probs = torch.softmax(sample_logits, dim=1)
    sample_pred = torch.max(sample_logits, 1)[1]

print("logits:")
print(sample_logits.cpu())

print("\nsoftmax probabilities:")
print(sample_probs.cpu())

print("\npred class:")
print(sample_pred.cpu())

print("\ntrue class:")
print(labels_val[indices_to_check].cpu())

해석:

- logits는 확률이 아니므로 합이 1이 아니다.
- softmax probabilities는 각 행의 합이 1이다.
- 가장 큰 확률의 index가 예측 class다.

## 22. 가중치 행렬과 bias 확인

다중 분류의 weight는 행렬이다.

```text
class 0을 위한 weight row
class 1을 위한 weight row
class 2를 위한 weight row
```

각 class마다 서로 다른 선형식이 있는 구조다.

In [ ]:
print("weight matrix:")
print(net.l1.weight.data.cpu())

print("\nbias vector:")
print(net.l1.bias.data.cpu())

> 필기 포인트:  
> 출력 class가 3개이므로 weight row도 3개다.  
> 각 row가 해당 class 점수를 계산하는 레시피라고 보면 된다.

## 23. 입력 변수 4개로 확장하기

이번에는 Iris의 feature 4개를 모두 사용한다.

2개 feature를 쓸 때보다 더 많은 정보를 사용할 수 있다.

```text
sepal length
sepal width
petal length
petal width
```

In [ ]:
x_all = x_org

x_train4, x_val4, y_train4, y_val4 = train_test_split(
    x_all,
    y_org,
    test_size=0.5,
    random_state=42,
    stratify=y_org
)

inputs4 = torch.tensor(x_train4).float().to(device)
labels4 = torch.tensor(y_train4).long().to(device)

inputs_val4 = torch.tensor(x_val4).float().to(device)
labels_val4 = torch.tensor(y_val4).long().to(device)

print("inputs4:", inputs4.shape)
print("labels4:", labels4.shape)

## 24. 4개 입력 모델 학습

입력 feature가 4개이므로 모델은 `nn.Linear(4, 3)` 구조가 된다.

In [ ]:
torch.manual_seed(42)

net4 = MultiClassLinearNet(n_input=4, n_output=3).to(device)

criterion4 = nn.CrossEntropyLoss()
optimizer4 = optim.SGD(net4.parameters(), lr=0.01)

history4 = train_linear_multiclass(
    net4,
    inputs4,
    labels4,
    inputs_val4,
    labels_val4,
    criterion4,
    optimizer4,
    num_epochs=1000,
    print_interval=50
)

print("2-feature final val loss:", history[-1, 3])
print("2-feature final val acc:", history[-1, 4])

print("4-feature final val loss:", history4[-1, 3])
print("4-feature final val acc:", history4[-1, 4])

In [ ]:
plt.plot(history[:, 0], history[:, 3], label="2 features val loss")
plt.plot(history4[:, 0], history4[:, 3], label="4 features val loss")
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.title("2 Features vs 4 Features")
plt.legend()
plt.show()

그래프 해석:

- feature를 늘리면 accuracy가 비슷해도 loss가 더 낮아질 수 있다.
- loss가 낮다는 것은 모델이 정답 class에 더 높은 확신을 줄 수 있다는 뜻이다.
- 단, feature가 많다고 항상 좋은 것은 아니며 noise feature가 많으면 과적합 위험도 있다.

## 25. Confusion Matrix와 Classification Report

다중 분류에서도 class별 성능을 봐야 한다.

In [ ]:
net4.eval()

with torch.no_grad():
    logits_val4 = net4(inputs_val4)
    pred_val4 = torch.max(logits_val4, 1)[1].cpu().numpy()

true_val4 = labels_val4.cpu().numpy()

cm_iris = confusion_matrix(true_val4, pred_val4)

print("[Classification Report]")
print(classification_report(true_val4, pred_val4, target_names=iris.target_names))

print("[Confusion Matrix]")
print(cm_iris)

In [ ]:
plt.imshow(cm_iris)
plt.title("Iris Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm_iris):
    plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar()
plt.show()

그래프 해석:

- 대각선 값은 맞힌 개수다.
- 대각선 밖의 값은 다른 class로 잘못 예측한 개수다.
- 어떤 class끼리 헷갈리는지 확인할 수 있다.

## 26. Wine 데이터로 다층 신경망 다중 분류

이번에는 Iris보다 feature가 많은 Wine 데이터를 사용한다.

Wine 데이터 구조는 다음이다.

```text
데이터 178개
feature 13개
class 3개
```

이 예제에서는 Hidden Layer, ReLU, Dropout, AdamW를 사용한다.

In [ ]:
wine = load_wine()

X_wine = wine.data
y_wine = wine.target

print("X_wine:", X_wine.shape)
print("y_wine:", y_wine.shape)
print("class names:", wine.target_names)
print("class counts:", np.bincount(y_wine))

## 27. Wine 데이터 분할과 표준화

train, validation, test로 나누고 feature를 표준화한다.

### 함수 사용법

```python
scaler.fit_transform(X_train)
scaler.transform(X_val)
scaler.transform(X_test)
```

- train에는 `fit_transform`을 사용한다.
- validation/test에는 train 기준으로 `transform`만 사용한다.
- validation/test에 `fit_transform`을 쓰면 데이터 누수 위험이 있다.

In [ ]:
X_temp, X_test_w, y_temp, y_test_w = train_test_split(
    X_wine,
    y_wine,
    test_size=0.2,
    random_state=42,
    stratify=y_wine
)

X_train_w, X_val_w, y_train_w, y_val_w = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

scaler_w = StandardScaler()

X_train_w = scaler_w.fit_transform(X_train_w)
X_val_w = scaler_w.transform(X_val_w)
X_test_w = scaler_w.transform(X_test_w)

print("Train:", X_train_w.shape)
print("Val:", X_val_w.shape)
print("Test:", X_test_w.shape)

## 28. Wine Tensor 변환

다중 분류이므로 label은 `long` 타입으로 변환한다.

In [ ]:
inputs_train_w = torch.tensor(X_train_w, dtype=torch.float32).to(device)
inputs_val_w = torch.tensor(X_val_w, dtype=torch.float32).to(device)
inputs_test_w = torch.tensor(X_test_w, dtype=torch.float32).to(device)

labels_train_w = torch.tensor(y_train_w, dtype=torch.long).to(device)
labels_val_w = torch.tensor(y_val_w, dtype=torch.long).to(device)
labels_test_w = torch.tensor(y_test_w, dtype=torch.long).to(device)

print("inputs_train_w:", inputs_train_w.shape, inputs_train_w.dtype)
print("labels_train_w:", labels_train_w.shape, labels_train_w.dtype)

## 29. 다층 신경망 모델 정의

다층 신경망은 선형 모델보다 표현력이 높다.

구조는 다음이다.

```text
Linear(13 → 64)
→ ReLU
→ Dropout
→ Linear(64 → 32)
→ ReLU
→ Dropout
→ Linear(32 → 3)
```

마지막 출력은 class 3개에 대한 logits다.

In [ ]:
class DeepMultiClassNet(nn.Module):
    def __init__(self, n_input, n_hidden1, n_hidden2, n_output, dropout_rate=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_input, n_hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(n_hidden2, n_output)
        )

    def forward(self, x):
        return self.network(x)

n_input_w = inputs_train_w.shape[1]
n_hidden1 = 64
n_hidden2 = 32
n_output_w = len(np.unique(y_wine))
dropout_rate = 0.3

model_wine = DeepMultiClassNet(
    n_input_w,
    n_hidden1,
    n_hidden2,
    n_output_w,
    dropout_rate
).to(device)

print(model_wine)

total_params = sum(p.numel() for p in model_wine.parameters())
trainable_params = sum(p.numel() for p in model_wine.parameters() if p.requires_grad)

print("total params:", total_params)
print("trainable params:", trainable_params)

### 함수 사용법 정리

```python
nn.Dropout(0.3)
```

- 학습 중 일부 뉴런 출력을 랜덤하게 꺼서 과적합을 줄인다.

```python
model.parameters()
```

- 모델 안의 학습 가능한 parameter를 Optimizer에게 넘길 때 사용한다.

## 30. Wine 평가 함수 만들기

평가 함수는 validation/test 단계에서 반복해서 쓴다.

In [ ]:
def evaluate_model(model, inputs, labels, criterion):
    model.eval()

    with torch.no_grad():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        pred = torch.max(outputs, 1)[1]
        acc = (pred == labels).float().mean()

    return loss.item(), acc.item()

> 기억할 점:  
> 평가할 때는 `model.eval()`과 `torch.no_grad()`를 함께 쓰는 습관이 좋다.  
> Dropout 같은 Layer는 train/eval 모드에 따라 동작이 달라진다.

## 31. Wine 학습 루프 만들기

다층 신경망은 AdamW Optimizer를 사용한다.

### 함수 사용법

```python
optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
```

- Adam 계열 Optimizer다.
- `weight_decay`는 L2 정규화 효과를 준다.

In [ ]:
criterion_wine = nn.CrossEntropyLoss()

learning_rate = 0.001

optimizer_wine = optim.AdamW(
    model_wine.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

print("criterion:", criterion_wine)
print("optimizer:", optimizer_wine.__class__.__name__)

In [ ]:
def train_model(model, inputs_train, labels_train, inputs_val, labels_val,
                criterion, optimizer, num_epochs=400, print_interval=20):
    history = []

    for epoch in range(num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        outputs = model(inputs_train)
        loss = criterion(outputs, labels_train)

        loss.backward()
        optimizer.step()

        if epoch % print_interval == 0:
            train_loss, train_acc = evaluate_model(model, inputs_train, labels_train, criterion)
            val_loss, val_acc = evaluate_model(model, inputs_val, labels_val, criterion)

            history.append([epoch, train_loss, train_acc, val_loss, val_acc])

    return np.array(history)

history_wine = train_model(
    model_wine,
    inputs_train_w,
    labels_train_w,
    inputs_val_w,
    labels_val_w,
    criterion_wine,
    optimizer_wine,
    num_epochs=400,
    print_interval=20
)

print("초기 val acc:", history_wine[0, 4])
print("최종 val acc:", history_wine[-1, 4])

## 32. Wine 학습 과정 시각화

Loss와 Accuracy를 확인한다.

In [ ]:
plt.plot(history_wine[:, 0], history_wine[:, 1], label="train loss")
plt.plot(history_wine[:, 0], history_wine[:, 3], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Wine Deep Model Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history_wine[:, 0], history_wine[:, 2], label="train acc")
plt.plot(history_wine[:, 0], history_wine[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Wine Deep Model Accuracy")
plt.legend()
plt.show()

그래프 해석:

- train loss와 validation loss가 함께 감소하면 학습이 안정적이다.
- train accuracy만 높고 validation accuracy가 낮으면 과적합을 의심한다.
- Dropout과 weight decay는 과적합을 줄이기 위한 장치다.

## 33. Wine 테스트 평가

학습이 끝난 뒤 test 데이터로 최종 성능을 확인한다.

In [ ]:
test_loss_wine, test_acc_wine = evaluate_model(
    model_wine,
    inputs_test_w,
    labels_test_w,
    criterion_wine
)

print("Test Loss:", test_loss_wine)
print("Test Accuracy:", test_acc_wine)

model_wine.eval()

with torch.no_grad():
    logits_test_wine = model_wine(inputs_test_w)
    pred_test_wine = torch.max(logits_test_wine, 1)[1].cpu().numpy()

true_test_wine = labels_test_w.cpu().numpy()

print("[Classification Report]")
print(classification_report(true_test_wine, pred_test_wine, target_names=wine.target_names))

## 34. 개별 샘플 예측 확인

개별 샘플에 대해 logits, softmax 확률, 예측 class를 확인한다.

In [ ]:
def predict_sample(model, inputs, labels, class_names, sample_idx=0):
    model.eval()

    with torch.no_grad():
        sample = inputs[sample_idx:sample_idx + 1]
        logits = model(sample)
        probs = torch.softmax(logits, dim=1)
        pred = torch.max(logits, 1)[1].item()
        true = labels[sample_idx].item()

    print("sample index:", sample_idx)
    print("logits:", logits.cpu().numpy().round(4))
    print("probabilities:", probs.cpu().numpy().round(4))
    print("predicted:", pred, class_names[pred])
    print("true:", true, class_names[true])

predict_sample(model_wine, inputs_test_w, labels_test_w, wine.target_names, sample_idx=3)

해석:

- logits는 모델의 raw score다.
- probabilities는 softmax를 적용한 확률 분포다.
- 가장 큰 확률을 가진 class가 최종 예측이다.

## 35. 큰 logit에서 Softmax를 직접 계산하면 위험한 이유

Softmax는 지수함수 `exp()`를 사용한다.

logit이 너무 크면 `exp(logit)`이 매우 커져 수치적으로 불안정해질 수 있다.

그래서 PyTorch는 `log_softmax`와 `CrossEntropyLoss`를 안정적으로 구현해둔다.

In [ ]:
big_logits = torch.tensor([[1000.0, 999.0, 995.0]])
target_big = torch.tensor([0])

stable_ce = F.cross_entropy(big_logits, target_big)
stable_log_softmax = F.log_softmax(big_logits, dim=1)

print("CrossEntropyLoss:", stable_ce.item())
print("stable log_softmax:")
print(stable_log_softmax)

> 필기 포인트:  
> `softmax → log`를 따로 계산하는 것보다 `log_softmax`를 쓰는 것이 안정적이다.  
> `CrossEntropyLoss`는 이 안정적인 처리를 내부에 포함한다.

## 36. 세 가지 구현 패턴 비교

다중 분류 손실 계산에는 여러 패턴이 있다.

| 패턴 | 모델 출력 | 손실 함수 | 권장 여부 |
|---|---|---|---|
| 패턴 1 | logits | CrossEntropyLoss | 권장 |
| 패턴 2 | log_softmax | NLLLoss | 가능 |
| 패턴 3 | softmax 후 log | NLLLoss | 비권장 |

가장 기본은 패턴 1이다.

In [ ]:
logits_pattern = torch.tensor([
    [2.0, 1.0, 0.0],
    [0.5, 0.2, 1.5]
])

labels_pattern = torch.tensor([0, 2])

loss_pattern1 = nn.CrossEntropyLoss()(logits_pattern, labels_pattern)

log_probs_pattern = F.log_softmax(logits_pattern, dim=1)
loss_pattern2 = nn.NLLLoss()(log_probs_pattern, labels_pattern)

probs_pattern = torch.softmax(logits_pattern, dim=1)
manual_log_probs = torch.log(probs_pattern)
loss_pattern3 = nn.NLLLoss()(manual_log_probs, labels_pattern)

print("Pattern 1 logits + CrossEntropyLoss:", loss_pattern1.item())
print("Pattern 2 log_softmax + NLLLoss:", loss_pattern2.item())
print("Pattern 3 softmax + log + NLLLoss:", loss_pattern3.item())

해석:

- 세 방식은 작은 예제에서는 거의 같은 값을 낸다.
- 하지만 수치 안정성 때문에 패턴 1 또는 패턴 2가 좋다.
- 실습과 실무에서는 `logits + CrossEntropyLoss`를 먼저 선택한다.

## 37. KL Divergence 개념

KL Divergence는 두 확률분포가 얼마나 다른지 재는 값이다.

```text
P: 실제 분포
Q: 모델이 예측한 분포
KL(P || Q): P 기준으로 Q가 얼마나 다른가
```

특징은 다음이다.

- 0에 가까울수록 두 분포가 비슷하다.
- 완전히 대칭은 아니다.
- 지식 증류, soft label, 분포 비교에서 자주 나온다.

In [ ]:
P = torch.tensor([[0.7, 0.2, 0.1]])
Q_good = torch.tensor([[0.65, 0.25, 0.10]])
Q_bad = torch.tensor([[0.10, 0.20, 0.70]])

kl_good = F.kl_div(torch.log(Q_good), P, reduction="batchmean")
kl_bad = F.kl_div(torch.log(Q_bad), P, reduction="batchmean")

print("KL(P || Q_good):", kl_good.item())
print("KL(P || Q_bad):", kl_bad.item())

### 함수 사용법: `F.kl_div()`

```python
F.kl_div(input_log_prob, target_prob, reduction="batchmean")
```

- 첫 번째 인자는 log probability다.
- 두 번째 인자는 target probability다.
- 그래서 `torch.log(Q)`를 넣는다.

> 주의:  
> `F.kl_div()`의 첫 번째 입력은 일반 확률이 아니라 log 확률이다.

## 38. KL Divergence와 Cross Entropy의 관계 감각

Cross Entropy와 KL Divergence는 둘 다 분포 차이를 다룬다.

간단히 감각만 잡으면 다음이다.

```text
Cross Entropy: 정답 분포 기준으로 예측 분포가 얼마나 잘 맞는가
KL Divergence: 두 분포가 얼마나 다른가
```

one-hot label 분류에서는 CrossEntropyLoss가 가장 자주 쓰인다.  
soft label이나 teacher-student 모델에서는 KL Divergence도 자주 등장한다.

In [ ]:
teacher_probs = torch.tensor([[0.80, 0.15, 0.05]])
student_logits = torch.tensor([[2.0, 1.0, 0.2]])

student_log_probs = F.log_softmax(student_logits, dim=1)

distillation_loss = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="batchmean"
)

print("teacher probs:", teacher_probs)
print("student log probs:", student_log_probs)
print("KL distillation style loss:", distillation_loss.item())

> 필기 포인트:  
> 일반적인 다중 분류 시험 문제에서는 CrossEntropyLoss를 먼저 기억하면 된다.  
> KL Divergence는 “분포와 분포를 비교하는 손실”로 기억하면 된다.

## 39. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `logits` | 모델 raw score | softmax 전 출력 |
| `Softmax` | 점수를 확률 분포로 변환 | `torch.softmax(logits, dim=1)` |
| `CrossEntropyLoss` | 다중 분류 표준 손실 | `nn.CrossEntropyLoss()` |
| `NLLLoss` | 정답 log probability에 음수 적용 | `nn.NLLLoss()` |
| `LogSoftmax` | softmax 후 log를 안정적으로 계산 | `F.log_softmax(x, dim=1)` |
| `KL Divergence` | 두 확률분포 차이 | `F.kl_div(log_q, p)` |
| `labels` | 정답 class index | dtype은 `torch.long` |
| `dim=1` | class 방향 | `[batch, class]`에서 class 축 |
| `torch.max(outputs, 1)` | 행별 최댓값과 index | 예측 class 추출 |
| `argmax(dim=1)` | 가장 큰 class index | `outputs.argmax(dim=1)` |
| `weight matrix` | class별 가중치 묶음 | `nn.Linear(in, out).weight` |
| `Dropout` | 일부 뉴런을 꺼 과적합 완화 | `nn.Dropout(0.3)` |
| `AdamW` | weight decay 포함 Adam 계열 Optimizer | `optim.AdamW(...)` |
| `StandardScaler` | 평균 0, 표준편차 1 표준화 | train에 fit, val/test에 transform |
| `Classification Report` | class별 성능 요약 | precision, recall, f1 |
| `Confusion Matrix` | class별 예측 오류 표 | `confusion_matrix()` |

## 40. 시험용 요약

```text
다중 분류 = 여러 class 중 하나를 고르는 문제
```

꼭 기억할 것:

- 이진 분류는 출력 1개, 다중 분류는 class 개수만큼 출력한다.
- 다중 분류 모델 출력 shape은 보통 `[batch, num_classes]`다.
- 출력 class가 N개이면 `nn.Linear(..., N)`을 사용한다.
- 이진 분류의 가중치는 벡터 느낌이고, 다중 분류의 가중치는 행렬이다.
- logits는 softmax 전 raw score다.
- Softmax는 logits를 확률 분포로 바꾼다.
- Softmax 결과는 각 행의 합이 1이다.
- `[batch, class]` 출력에서는 `softmax(dim=1)`을 사용한다.
- PyTorch 표준 패턴은 `logits + CrossEntropyLoss`다.
- `CrossEntropyLoss`에 Softmax를 먼저 적용하지 않는다.
- `CrossEntropyLoss`는 내부적으로 LogSoftmax + NLLLoss를 처리한다.
- 다중 분류 정답 label은 one-hot이 아니라 class index다.
- `CrossEntropyLoss`의 label dtype은 `torch.long`이어야 한다.
- 예측 class는 `torch.max(outputs, 1)[1]`로 구한다.
- `torch.max(outputs, 1)`은 values와 indices를 반환한다.
- 우리가 필요한 예측 label은 indices다.
- NLLLoss는 정답 class의 log probability만 뽑아 loss를 계산한다.
- `LogSoftmax + NLLLoss`는 `CrossEntropyLoss`와 같은 원리다.
- 직접 `Softmax + log`를 쓰는 방식은 수치적으로 불안정할 수 있다.
- 입력 feature를 늘리면 accuracy가 같아도 loss가 낮아질 수 있다.
- Dropout은 과적합을 줄이는 데 도움을 준다.
- KL Divergence는 두 확률분포가 얼마나 다른지 재는 값이다.